# TARDIS_MODEL

## Importation des modules nécessaires

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## Chargement de la donnée 

In [2]:
# Lecture du dataset nettoyé
df = pd.read_csv("cleaned_dataset.csv", sep=";")

# Taille du dataset
print(f"Nombre d'échantillons: {df.shape[0]}")
print(f"Nombre de caractéristiques: {df.shape[1]}")

# Aperçu du dataset
print("\nAperçu du dataset:")
df.head()

Nombre d'échantillons: 12070
Nombre de caractéristiques: 26

Aperçu du dataset:


,Date,Service,Departure station,Arrival station,Average journey time,Number of scheduled trains,Number of cancelled trains,Cancellation comments,Number of trains delayed at departure,Average delay of late trains at departure,...,Number of trains delayed > 15min,Average delay of trains > 15min (if competing with flights),Number of trains delayed > 30min,Number of trains delayed > 60min,Pct delay due to external causes,Pct delay due to infrastructure,Pct delay due to traffic management,Pct delay due to rolling stock,Pct delay due to station management and equipment reuse,"Pct delay due to passenger handling (crowding, disabled persons, connections)"
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.0,870,5.0,NaN,289.0,11.24780854,...,110.0,6.51,44.0,8.0,36.13445378,31.09243697,10.92436975,15.96638655,"5,04",0.840336134
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56.0,406.0,1.0,NaN,213.0,8.479968701,...,32.0,5.363539095,9.0,4.0,20.0,35.0,16.66666667,16.66666667,8.333333333,3.333333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.0,226.0,0.0,NaN,21.0,6.23968254,...,11.0,2.938053097,6.0,1.0,22.22222222,27.77777778,16.66666667,16.66666667,5.555555556,11.11111111
3,2018-01,National,PARIS MONTPARNASSE,NANTES,216.21,508.0,3.0,NaN,71.0,7.235211268,...,39.0,5.292211221,18.0,NaN,33.33333333,22.22222222,16.66666667,20.37037037,5.555555556,1.851851852
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94.0,472.0,4.0,NaN,224.0,6.784672619,...,42.0,4.882371795,10.0,0.0,15.78947368,45.61403509,NaN,15.78947368,1.754385965,1.754385965


In [3]:
# Informations sur le dataset
print("Informations sur le dataset:")
df.info()

Informations sur le dataset:
<class 'pandas.DataFrame'>
RangeIndex: 12070 entries, 0 to 12069
Data columns (total 26 columns):
 #   Column                                                                         Non-Null Count  Dtype
---  ------                                                                         --------------  -----
 0   Date                                                                           12010 non-null  str  
 1   Service                                                                        11830 non-null  str  
 2   Departure station                                                              12011 non-null  str  
 3   Arrival station                                                                12011 non-null  str  
 4   Average journey time                                                           11830 non-null  str  
 5   Number of scheduled trains                                                     11830 non-null  str  
 6   Number of cancelled 

## Cleaning and Conversion

Cette partie ne devrait pas normalement exister. Mais au vue du retard des autres membres de l'équipe, je me vois obliger de faire un petit truc pour pouvoir tester des modèles. Il s'agira donc de sélectionner les colonnes numériques, de les convertir puis de supprimer les lignes contenant une ou plusieurs valeurs manquantes au niveau de ces colonnes.

In [4]:
# Liste de quelques colonnes numériques
cols_numeriques = [
    "Number of scheduled trains",
    "Number of cancelled trains",
    "Number of trains delayed at departure",
    "Average delay of all trains at arrival"
]

# Conversion des colonnes en type numériques
for col in cols_numeriques:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [5]:
# Reduction du nombre de colonnes
df = df[cols_numeriques]

# Taille du nouveau dataset
print(f"Nombre d'échantillons: {df.shape[0]}")
print(f"Nombre de caractéristiques: {df.shape[1]}")

# Aperçu du nouveau dataset
print("\nAperçu du nouveau dataset:")
df.head()

Nombre d'échantillons: 12070
Nombre de caractéristiques: 4

Aperçu du nouveau dataset:


,Number of scheduled trains,Number of cancelled trains,Number of trains delayed at departure,Average delay of all trains at arrival
0,870.0,5.0,289.0,6.511118
1,406.0,1.0,213.0,5.363539
2,226.0,0.0,21.0,2.938053
3,508.0,3.0,71.0,5.292211
4,472.0,4.0,224.0,4.882372


In [6]:
# Nombre de valeurs manquantes
df.isnull().sum()

Number of scheduled trains                266
Number of cancelled trains                266
Number of trains delayed at departure     260
Average delay of all trains at arrival    366
dtype: int64

In [7]:
# Suppression des valeurs manquantes
df = df.dropna()

In [8]:
# Taille du dataset après suppression des valeurs manquantes
print(f"Nombre d'échantillons: {df.shape[0]}")
print(f"Nombre de caractéristiques: {df.shape[1]}")

Nombre d'échantillons: 10963
Nombre de caractéristiques: 4


## Choix des variables

In [9]:
# Définition de X (features) et y (target)
X = df[["Number of scheduled trains",
        "Number of cancelled trains",
        "Number of trains delayed at departure"]]
y = df["Average delay of all trains at arrival"]

# Division en train et test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

## Entrainement et Evaluation

In [10]:
# Entrainement
model = LinearRegression()
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Evaluation
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE\t:{rmse: .2f} minutes")
print(f"MAE\t:{mae: .2f} minutes")
print(f"R2\t:{r2: .4f}")

RMSE	: 5.78 minutes
MAE	: 3.04 minutes
R2	: 0.0037


## Sauvegarde du modèle

In [11]:
joblib.dump(model, "model.joblib")
print("Modèle sauvegardé avec succès")

Modèle sauvegardé avec succès
